# Od napovedi do poslovne odločitve — A: prikaz (telekom)
**Akademija UI za poslovne aplikacije — Modul 2, Uvod v strojno učenje**

Ta zvezek poganja predavatelj. Vajo naredite v zvezku **`B_poslovna_odlocitev`**.

Vprašanje: **pokličemo lahko 100 strank. Katere izberemo?**

### Ponovitev
- Model vsaki stranki pripiše **verjetnost odhoda** (`predict_proba`).
- Pri omejenem proračunu (100 klicev) strank ne delimo s pragom 0,5. **Razvrstimo** jih po verjetnosti in vzamemo vrh seznama.
- Isto vprašanje zastavimo vsakemu modelu: **koliko od izbranih 100 strank je res odšlo?** Tako dobimo lestvico.
- **Tveganje ni isto kot korist:** visoka verjetnost odhoda ne pove, ali bi naša ponudba stranko zadržala.
- **Mere iz prejšnjega sklopa** (točnost, natančnost, priklic) ocenijo vse napovedi pri pragu 0,5. Pri 100 klicih šteje samo vrh seznama.

### Nastavitev okolja
Zaženite spodnji dve celici (**Shift + Enter**). Ničesar ne spreminjajte. Koda prve celice je skrita.
Druga celica pokliče skupno pripravo podatkov iz `utils.modeliranje`. Veljavne pripravljene tabele ponovno uporabi.

In [ ]:
#@title Nastavitev okolja (kliknite za prikaz kode)
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/valira-ai/ui-za-poslovne-aplikacije.git"

if "google.colab" in sys.modules:
    KOREN = Path("/content/ui-za-poslovne-aplikacije")
    if not (KOREN / "utils/orodja.py").is_file():
        if KOREN.exists():
            raise FileNotFoundError(f"Mapa {KOREN} že obstaja, vendar v njej ni utils/orodja.py.")
        subprocess.run(["git", "clone", "-q", REPO_URL, str(KOREN)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(KOREN / "requirements.txt")],
        check=True,
    )
else:
    KOREN = Path.cwd().resolve()
    while not (KOREN / "utils/orodja.py").is_file() and KOREN != KOREN.parent:
        KOREN = KOREN.parent
    if not (KOREN / "utils/orodja.py").is_file():
        raise FileNotFoundError("Zaženite zvezek v prenesenem repozitoriju ali njegovi podmapi (manjka utils/orodja.py).")

if str(KOREN) not in sys.path:
    sys.path.insert(0, str(KOREN))
print("Gradivo je v:", KOREN)


In [ ]:
from utils.modeliranje import poskrbi_za_modeliranje, preberi_shemo, preberi_tabelo
PODATKI = poskrbi_za_modeliranje("telekom")
shema = preberi_shemo(PODATKI / "shema.json")


In [ ]:
import numpy as np
import pandas as pd
from plotnine import *
from sklearn.tree import DecisionTreeClassifier

from utils.orodja import nastavi_slog, MODRA, ORANZNA, ZELENA, SIVA, SEME, TELEKOM_STEVILSKE
from utils.orodja import mere
nastavi_slog()

<details>
<summary><b>Kako so pripravljeni podatki? (klikni)</b></summary>

Podatke je pripravila skripta `scripts/pripravi_telekom.py`. Naredi isto kot zvezka predavatelja
`A1_Priprava_Telekom` (mapa `03_Uvod_v_strojno_ucenje/notebooks/01_priprava_podatkov`) in
`A1_Delitev_in_manjkajoce_Telekom` (mapa `03_Uvod_v_strojno_ucenje/notebooks/02_preverjanje_modela`).
Tam so vse odločitve z razlogi. Na kratko:

1. Ena vrstica = ena stranka (100 000 strank). Cilj je `odhod`. `id_stranke` ni značilka.
2. Tabela ima 64 značilk: 58 številskih in 6 besedilnih. Stolpcem iz 1. modula so dodani novi
   (3- in 6-mesečna povprečja, razmerja, logaritmi).
3. Skrite oznake (`U`, `UNKW`) in nemogoče negativne vrednosti so postale manjkajoče vrednosti.
4. Delitev 60 / 20 / 20: naključna, z enakim deležem odhodov, seme 42. Vsi imamo iste stranke v istih množicah.
5. Manjkajoče številke so zapolnjene z **mediano učne množice**. Stolpci `..._manjka` povedo, kje je vrednost manjkala.
6. Besedilni stolpci so zapisani tudi kot stolpci 0/1 (na primer `regija_CHICAGO_AREA`).
   Skupaj je 96 stolpcev, ki jih lahko podamo modelu.
7. Osamelcev ne odstranimo in značilk ne skaliramo: drevo vpraša samo, ali je vrednost nad ali pod pragom.
8. Uporabimo 16 številskih značilk iz 1. modula, kot pri drevesu. Z istimi značilkami bodo na lestvico prišli tudi naslednji modeli.
9. Modele primerjamo na validacijski množici. Testna ostane zaprta do tekmovanja na koncu klasifikacijskega dela.

</details>

In [ ]:
ucni = preberi_tabelo(PODATKI / "train.csv", shema)
validacijski = preberi_tabelo(PODATKI / "val.csv", shema)

ZNACILKE = TELEKOM_STEVILSKE                                           # 16 številskih značilk iz 1. modula
X_ucni, y_ucni = ucni[ZNACILKE], ucni["odhod"]
X_val, y_val = validacijski[ZNACILKE], validacijski["odhod"]
print(f"Učnih strank: {len(ucni)}, validacijskih: {len(validacijski)}, od teh je odšlo {y_val.mean():.1%}")
print(f"Stolpcev v tabeli: {ucni.shape[1]}, manjkajočih vrednosti: {int(ucni.isna().sum().sum())}")

### Model iz prejšnjega sklopa
Drevo globine 5 naučimo na istih 16 številskih značilkah kot v sklopu o drevesu. Skaliranja ne potrebuje:
vrednost samo primerja s pragom.

In [ ]:
drevo = DecisionTreeClassifier(max_depth=5, random_state=SEME).fit(X_ucni, y_ucni)

# verjetnost odhoda za vsako validacijsko stranko
verjetnosti_drevo = pd.Series(drevo.predict_proba(X_val)[:, 1], index=X_val.index)
print(f"Drevo je naučeno. Validacijska točnost: {drevo.score(X_val, y_val):.3f}")

---
## A1. Od verjetnosti do seznama
Vsaka validacijska stranka ima verjetnost odhoda. Razvrstimo jih od največje do najmanjše.

In [ ]:
print("Različnih verjetnosti pri drevesu:", verjetnosti_drevo.nunique())
verjetnosti_drevo.sort_values(ascending=False).head(8)

Drevo globine 5 ima malo listov, zato ima veliko strank **enako** verjetnost. Katere od njih pridejo med prvih 100?
Potrebujemo jasen dogovor: **stranke z enako oceno razvrstimo naključno, vedno z istim semenom.**
Tako vsi dobimo enak rezultat in nobeno pravilo nima prednosti zaradi vrstnega reda v tabeli.

In [ ]:
def izberi_najbolj_tvegane(ocene, k=100):
    """Vrne k strank z najvišjo oceno. Stranke z enako oceno razvrstimo naključno (seme 42)."""
    premesane = ocene.dropna().sample(frac=1, random_state=SEME)
    return premesane.sort_values(ascending=False, kind="stable").index[:k]

izbrane = izberi_najbolj_tvegane(verjetnosti_drevo, 100)
print("Od 100 izbranih strank jih je res odšlo:", int(y_val.loc[izbrane].sum()))

## A2. Je to veliko? Primerjava z naključnim izborom
Če 100 strank izberemo naključno, jih v povprečju odide toliko, kot je delež odhodov: približno 50.
Posamezen naključen izbor niha — spodaj je en primer.

Vsaka pika je ena izbrana stranka: oranžna je odšla, modra je ostala.

In [ ]:
def graf_izbranih(izbori):
    """izbori = {naslov: odhod izbranih 100 strank}. Za vsak izbor mreža 10 x 10: oranžna = odšla."""
    deli = []
    for naslov, odhod in izbori.items():
        urejene = odhod.sort_values(ascending=False).reset_index(drop=True)
        deli.append(pd.DataFrame({"x": urejene.index % 10, "y": 9 - urejene.index // 10,
                                  "stranka": urejene.map({1: "odšla", 0: "ostala"}),
                                  "izbor": f"{naslov}\nodšlo: {int(urejene.sum())} od {len(urejene)}"}))
    mreza = pd.concat(deli)
    mreza["izbor"] = pd.Categorical(mreza["izbor"], mreza["izbor"].unique())     # vrstni red kot v slovarju
    return (ggplot(mreza, aes(x="x", y="y", color="stranka"))
            + geom_point(size=7.5)
            + scale_color_manual(values={"odšla": ORANZNA, "ostala": MODRA})
            + facet_wrap("~izbor")
            + coord_equal()
            + labs(color="")
            + theme_void()
            + theme(figure_size=(0.9 + 3.9 * len(izbori), 4.5),
                    strip_text=element_text(size=11, weight="bold", linespacing=1.4)))

nakljucne = y_val.sample(100, random_state=7)          # en naključen izbor 100 strank
print(f"Pričakovano pri naključnem izboru: {100 * y_val.mean():.0f} od 100")

graf_izbranih({"Naključnih 100 strank": nakljucne, "100 strank po drevesu": y_val.loc[izbrane]})

## A3. Lestvica
Isto naredimo za vsak model. Poleg 100 strank pogledamo še **1 000 strank**: večja skupina je manj odvisna od naključja.
Funkcija `vrstica_lestvice` uporabi isti izbor kot zgoraj (kopija je v `utils/orodja.py`).

Število strank, ki so med izbranimi 100 res odšle, je **natančnost** na teh 100 strankah: 86 od 100 je natančnost 0,86.

In [ ]:
def vrstica_lestvice(ime, ocene, odhod):
    """Koliko od izbranih 100 oziroma 1 000 strank je res odšlo."""
    return {"izbira": ime,
            "odšlo od 100": int(odhod.loc[izberi_najbolj_tvegane(ocene, 100)].sum()),
            "odšlo od 1 000": int(odhod.loc[izberi_najbolj_tvegane(ocene, 1000)].sum())}

lestvica = pd.DataFrame([
    {"izbira": "naključno (pričakovano)", "odšlo od 100": round(100 * y_val.mean()), "odšlo od 1 000": round(1000 * y_val.mean())},
    vrstica_lestvice("drevo (globina 5)", verjetnosti_drevo, y_val),
])
lestvica

In [ ]:
(ggplot(lestvica, aes(x="izbira", y="odšlo od 100"))
 + geom_col(fill=MODRA)
 + geom_text(aes(label="odšlo od 100"), ha="left", nudge_y=1.5, size=11)
 + scale_x_discrete(limits=lestvica["izbira"].tolist())
 + scale_y_continuous(limits=(0, 100))
 + coord_flip()
 + labs(title="Lestvica: koliko od 100 izbranih strank je res odšlo?", x="", y="odšlo od 100")
 + theme(figure_size=(8, 2.8)))

To lestvico bomo dopolnjevali ves dan: vsak nov model (logistična regresija, gozd, pospešena drevesa) dobi svojo vrstico. Vsi modeli so ocenjeni na
**istih validacijskih strankah** in z **istimi značilkami**. Lestvica je vaja na približno uravnoteženem izvlečku;
za resnično podjetje ne poznamo deleža odhodov niti obdobja, v katerem je bil odhod označen.

## A4. Točnost ne pove vsega
V sklopu o drevesu je imelo najboljšo validacijsko točnost drevo **globine 8** (0,606). Je zato tudi najboljše za 100 klicev?
Obe drevesi naučimo na istih učnih strankah in ju ocenimo na istih validacijskih.

In [ ]:
drevo_8 = DecisionTreeClassifier(max_depth=8, random_state=SEME).fit(X_ucni, y_ucni)
verjetnosti_drevo_8 = pd.Series(drevo_8.predict_proba(X_val)[:, 1], index=X_val.index)

vrstice = []
for ime, model, verjetnosti in [("drevo (globina 5)", drevo, verjetnosti_drevo), ("drevo (globina 8)", drevo_8, verjetnosti_drevo_8)]:
    m = mere(y_val, model.predict(X_val))                       # vse stranke, prag 0,5
    vrstice.append({"model": ime, "točnost": m["točnost"], "natančnost": m["natančnost"], "priklic": m["priklic"],
                    **{k: v for k, v in vrstica_lestvice(ime, verjetnosti, y_val).items() if k != "izbira"}})
primerjava = pd.DataFrame(vrstice).set_index("model")
primerjava.round(3)

In [ ]:
dolga = pd.DataFrame({"model": primerjava.index.tolist() * 2,
                      "mera": ["točnost (vseh 20 000 strank)"] * 2 + ["odšlo od 100 izbranih"] * 2,
                      "vrednost": primerjava["točnost"].tolist() + primerjava["odšlo od 100"].tolist()})
dolga["oznaka"] = [f"{v:.3f}" if v < 1 else f"{v:.0f}" for v in dolga["vrednost"]]
dolga["mera"] = pd.Categorical(dolga["mera"], ["točnost (vseh 20 000 strank)", "odšlo od 100 izbranih"])     # vrstni red grafov

(ggplot(dolga, aes(x="model", y="vrednost", fill="model"))
 + geom_col(width=0.6, show_legend=False)
 + geom_text(aes(label="oznaka"), va="top", color="white", size=12)                          # številka na vrhu stolpca
 + facet_wrap("~mera", scales="free_y")
 + scale_fill_manual(values=[MODRA, ORANZNA])
 + labs(title="Globina 8 zmaga po točnosti, globina 5 pri 100 klicih", x="", y="")
 + theme(figure_size=(8, 3.4)))

- Po **točnosti** in **priklicu** je boljše drevo globine 8 (0,606 proti 0,596).
- Med **prvimi 100** strankami jih pri globini 8 odide samo 72, pri globini 5 pa 86.
- Pri prvih 1 000 sta drevesi skoraj enaki (713 in 720).

Točnost, natančnost in priklic ocenijo **vse** stranke pri pragu 0,5. Pri 100 klicih pa šteje samo **vrh seznama**:
ali so stranke z največjo verjetnostjo res najbolj tvegane. Zakaj je globlje drevo tu slabše, raziščete v zvezku B (Za hitre).

Mero izberemo po **poslovnem vprašanju**, ne po navadi. Model z najboljšo točnostjo ni nujno najboljši za našo odločitev.

## A5. Tveganje ni isto kot korist
Lestvica pove, **kdo bo verjetno odšel**. Ne pove, **koga lahko zadržimo**:
- nekatere stranke bi odšle ne glede na ponudbo,
- nekatere bi ostale tudi brez nje (ponudba je strošek),
- podatkov o tem, kako stranke reagirajo na ponudbo, v tej tabeli nimamo.

Za odgovor na to vprašanje bi potrebovali poskus: delu strank ponudbo pošljemo, delu ne, in primerjamo.

> **Povezava s 1. modulom:** korelacija ni vzročnost. Model najde stranke, pri katerih so značilke *povezane* z odhodom.
> Ne pove, kaj odhod *povzroča* in kaj bi ga preprečilo.

---
Naprej: zvezek **`B_poslovna_odlocitev`** — poskusite model premagati s svojim pravilom.